# Hypothesis testing — Bitcoin and S&P 500 returns

Do monthly Bitcoin and S&P 500 returns show a linear association over the study period?
The calendar and return definitions come from `02_market_data_validation.ipynb`.
Use the CSV produced by that notebook and run this notebook from the same directory.

In the linear projection of Bitcoin returns onto S&P 500 returns,
$\beta = \operatorname{Cov}(X,Y)/\operatorname{Var}(X)$, where $X$ is the S&P 500 return
and $Y$ is the Bitcoin return. With positive variances, zero slope is equivalent to zero
Pearson correlation. We test $H_0: \beta=0$ against $H_1: \beta\ne0$, with $\alpha=0.05$.

We chose the hypothesis after inspecting these data, so the inference is exploratory.
The model summarizes same-month co-movement. It cannot identify causal effects or
establish forecasting performance.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import norm

data_path = Path("../data/processed/market_monthly_2018_2026.csv")
return_columns = ["sp500_return_pct", "btc_return_pct"]
monthly = pd.read_csv(data_path, usecols=["month", *return_columns])
monthly["month"] = pd.to_datetime(monthly["month"], format="%Y-%m-%d").dt.to_period("M")
paired = monthly.set_index("month").sort_index().dropna(subset=return_columns)

lags = (3, 6, 12)
if len(paired) <= max(lags):
    raise ValueError("The sample needs more complete months than the largest HAC lag.")

expected = pd.period_range(paired.index[0], paired.index[-1], freq="M")
if not paired.index.equals(expected):
    raise ValueError("Paired returns contain duplicate months or internal calendar gaps.")

x, y = paired[return_columns].to_numpy(dtype=float).T
if not np.isfinite([x, y]).all() or min(np.std(x), np.std(y)) == 0:
    raise ValueError("Both return series must be finite and have positive variance.")

print(f"{len(paired)} paired months, {paired.index[0]} to {paired.index[-1]}")

105 paired months, 2018-02 to 2026-10


## Regression and uncertainty

Fit $Y_t = a + \beta X_t + u_t$ by ordinary least squares. With centered predictor
$\tilde X_t = X_t-\bar X$ and residuals $\hat u_t$, the slope score is
$s_t = \tilde X_t\hat u_t$. The Newey–West variance estimate for the slope is

$$
\widehat{\operatorname{Var}}(\hat\beta) =
\frac{\sum_t s_t^2 +
2\sum_{h=1}^{L}\left(1-\frac{h}{L+1}\right)\sum_{t=h+1}^{n}s_t s_{t-h}}
{\left(\sum_t \tilde X_t^2\right)^2}.
$$

The Bartlett weights taper the contribution of more distant lags. The
heteroskedasticity- and autocorrelation-consistent (HAC) standard error gives an
asymptotic normal test and confidence interval. We use six monthly lags for the
main estimate and three and twelve for sensitivity on the same sample.

This approximation requires a sufficiently stable linear projection and weak
temporal dependence. The sample size and variation in rolling correlations
in notebook 02 limit the precision and scope of the inference.

In [2]:
x_centered = x - x.mean()
sum_x_squared = np.dot(x_centered, x_centered)
slope = np.dot(x_centered, y - y.mean()) / sum_x_squared
residuals = y - y.mean() - slope * x_centered
scores = x_centered * residuals

alpha = 0.05
critical_value = norm.isf(alpha / 2)
rows = []

for lag_count in lags:
    score_sum = np.dot(scores, scores)
    for lag in range(1, lag_count + 1):
        weight = 1 - lag / (lag_count + 1)
        score_sum += 2 * weight * np.dot(scores[lag:], scores[:-lag])

    if not np.isfinite(score_sum) or score_sum <= 0:
        raise ValueError("The estimated slope variance must be positive and finite.")

    standard_error = np.sqrt(score_sum) / sum_x_squared
    z_statistic = slope / standard_error
    margin = critical_value * standard_error
    rows.append({
        "HAC lags": lag_count,
        "Slope": slope,
        "Standard error": standard_error,
        "95% CI lower": slope - margin,
        "95% CI upper": slope + margin,
        "Two-sided p-value": 2 * norm.sf(abs(z_statistic)),
    })

results = pd.DataFrame(rows).set_index("HAC lags")
print(f"Pearson correlation: {np.corrcoef(x, y)[0, 1]:.3f}")
formatted_results = results.round(3)
formatted_results["Two-sided p-value"] = results["Two-sided p-value"].map("{:.4g}".format)
display(formatted_results)

Pearson correlation: 0.400


,Slope,Standard error,95% CI lower,95% CI upper,Two-sided p-value
HAC lags,,,,,
3,1.633,0.344,0.958,2.308,2.109e-06
6,1.633,0.325,0.997,2.270,4.956e-07
12,1.633,0.357,0.934,2.333,4.744e-06


In [3]:
main = results.loc[6]
p_value = main["Two-sided p-value"]
decision = "Reject" if p_value < alpha else "Do not reject"
print(f"{decision} the zero-slope hypothesis at the 5% level (p = {p_value:.4g}).")
print(
    f"A one-percentage-point increase in S&P 500 return corresponds to a "
    f"{slope:+.3f} percentage-point difference in fitted Bitcoin return "
    f"(95% CI [{main['95% CI lower']:.3f}, {main['95% CI upper']:.3f}])."
)
decisions_agree = (results["Two-sided p-value"] < alpha).nunique() == 1
print(f"The test decision {'agrees' if decisions_agree else 'varies'} across the three lag choices.")

Reject the zero-slope hypothesis at the 5% level (p = 4.956e-07).
A one-percentage-point increase in S&P 500 return corresponds to a +1.633 percentage-point difference in fitted Bitcoin return (95% CI [0.997, 2.270]).
The test decision agrees across the three lag choices.


## Conclusion

Across 105 paired months from February 2018 through October 2026,
Bitcoin and S&P 500 monthly returns showed a positive linear
association (Pearson r = 0.400).

In the main regression with six Newey–West lags, a one-percentage-point
higher S&P 500 return was associated with a 1.633-percentage-point
higher fitted Bitcoin return. The 95% confidence interval for the slope
was [0.997, 2.270], and the two-sided p-value was 4.956 × 10⁻⁷.
The slope intervals also excluded zero with three and twelve lags.

In notebook 02, the 95% moving-block bootstrap intervals for Pearson
correlation were above zero for all three block lengths. The two methods
estimate different quantities on the same sample. Their agreement supports
a positive association across the methods and settings examined.

We formed the hypothesis after inspecting the data, so the inference is
exploratory. The results describe same-month association over the study
period. Establishing causation or assessing forecasting performance would
require a different study design.
